# Interactive River Network Map — Compressed ⇄ Uncompressed

A version of `Interactive_River_Network_Map.ipynb` that also shows the **uncompressed** Colorado
river network (`largest_river_component.pkl`, one node per MERIT river pixel) and lets you
**toggle between the compressed and uncompressed graphs** on the site.

**Compressed view** — same as the original map: 67 retained nodes, edges drawn along the real river path,
WQ training/test stations, hydrology support, sources, outlet, junctions.

**Uncompressed view** — every river-pixel node (~67k). These are drawn on a canvas only for the
area you are looking at once you zoom in (zoom ≥ 11), so the page stays fast. Click any pixel
to see its raster row/col, distance to the outlet, number of upstream pixels and which compressed
edge it was folded into. Also shows WQ stations snapped to the pixel graph.

The toggle sits in the top-right corner; you can also link straight to a view with
`index.html#uncompressed` or `index.html#compressed`.

### Files used
- `graphs/colorado/master_compressed_river_graph.pkl`
- `graphs/colorado/largest_river_component.pkl`

### Output
- `figures/interactive_river_network_map_toggle.html`
- `index.html` (the published site)

In [ ]:
import os
import json
import math
import pickle
import html
from getpass import getpass

import numpy as np
import pandas as pd
import networkx as nx
import folium
from folium import plugins
from branca.element import Element, MacroElement
from jinja2 import Template

MASTER_GRAPH_FILE = "graphs/colorado/master_compressed_river_graph.pkl"
DETAILED_GRAPH_FILE = "graphs/colorado/largest_river_component.pkl"

OUTPUT_HTML = "figures/interactive_river_network_map_toggle.html"
SITE_HTML = "index.html"          # set to None to skip updating the site
OUTPUT_POINTS_CSV = "data/colorado/interactive_river_map_points_toggle.csv"

# Uncompressed pixel nodes are only drawn at/after this zoom level
PIXEL_MIN_ZOOM = 11
PIXEL_MAX_IN_VIEW = 8000

print("Working directory:", os.getcwd())
print("Master graph exists:", os.path.exists(MASTER_GRAPH_FILE))
print("Detailed graph exists:", os.path.exists(DETAILED_GRAPH_FILE))

## 1. Load both graphs

In [ ]:
with open(MASTER_GRAPH_FILE, "rb") as f:
    G = pickle.load(f)            # compressed

with open(DETAILED_GRAPH_FILE, "rb") as f:
    G_detail = pickle.load(f)     # uncompressed (one node per river pixel)

for name, graph in [("Compressed", G), ("Uncompressed", G_detail)]:
    print(f"{name}: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges, "
          f"weak components = {nx.number_weakly_connected_components(graph)}")

missing = [n for n in G if n not in G_detail]
print("Compressed nodes missing from detailed graph:", len(missing))

## 2. Classify structural nodes and monitoring roles

In [ ]:
def structural_nodes(graph):
    sources, outlets, junctions = set(), set(), set()
    for node in graph:
        indeg, outdeg = graph.in_degree(node), graph.out_degree(node)
        if indeg == 0 and outdeg > 0:
            sources.add(node)
        if indeg > 0 and outdeg == 0:
            outlets.add(node)
        if indeg + outdeg >= 3:
            junctions.add(node)
    return sources, outlets, junctions


source_nodes, outlet_nodes, junction_nodes = structural_nodes(G)
d_source_nodes, d_outlet_nodes, d_junction_nodes = structural_nodes(G_detail)

train_nodes, full_test_nodes, partial_test_nodes, support_nodes = set(), set(), set(), set()

for node, data in G.nodes(data=True):
    for cluster in data.get("usgs_clusters", []):
        role = cluster.get("ml_role")
        if role == "train":
            train_nodes.add(node)
        elif role == "test_full":
            full_test_nodes.add(node)
        elif role == "test_partial":
            partial_test_nodes.add(node)
    if data.get("support_gauge_count", 0) > 0:
        support_nodes.add(node)

# WQ stations snapped directly onto the pixel graph (from main.ipynb)
snapped_station_nodes = {n for n, d in G_detail.nodes(data=True) if d.get("station_count", 0) > 0}

print(f"{'':28s}{'compressed':>12s}{'uncompressed':>14s}")
print(f"{'Network sources':28s}{len(source_nodes):12d}{len(d_source_nodes):14d}")
print(f"{'Outlets':28s}{len(outlet_nodes):12d}{len(d_outlet_nodes):14d}")
print(f"{'Junctions':28s}{len(junction_nodes):12d}{len(d_junction_nodes):14d}")
print()
print("Training 5/5 locations:", len(train_nodes))
print("Held-out 5/5 test locations:", len(full_test_nodes))
print("Partial WQ test locations:", len(partial_test_nodes))
print("Hydrology support nodes:", len(support_nodes))
print("Pixel nodes with snapped WQ stations:", len(snapped_station_nodes))

## 3. Measure downstream source coverage (compressed graph)

In [ ]:
measurement_nodes = {
    node for node, data in G.nodes(data=True)
    if data.get("usgs_station_count", 0) > 0 or data.get("support_gauge_count", 0) > 0
}

source_coverage_rows = []
for source in source_nodes:
    lengths = nx.single_source_dijkstra_path_length(G, source, weight="length_m")
    reachable = {n: d for n, d in lengths.items() if n in measurement_nodes}
    if reachable:
        nearest = min(reachable, key=reachable.get)
        distance_km = reachable[nearest] / 1000
    else:
        nearest, distance_km = None, np.nan
    source_coverage_rows.append({
        "source_node": source,
        "nearest_measurement_node": nearest,
        "distance_to_first_measurement_km": distance_km,
    })

source_coverage = pd.DataFrame(source_coverage_rows)
source_distance_lookup = (
    source_coverage.set_index("source_node")["distance_to_first_measurement_km"].to_dict()
    if len(source_coverage) else {}
)
print(source_coverage["distance_to_first_measurement_km"].describe())

## 4. Uncompressed pixel attributes

For every pixel node: distance to the outlet along the river, number of upstream pixels,
and which compressed edge (via `original_path`) it was folded into.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    r = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def pixel_step_m(a, b):
    na, nb = G_detail.nodes[a], G_detail.nodes[b]
    return haversine_m(na["latitude"], na["longitude"], nb["latitude"], nb["longitude"])


topo = list(nx.topological_sort(G_detail))

# Distance to outlet (walk downstream order in reverse)
dist_to_outlet_m = {}
for node in reversed(topo):
    succ = list(G_detail.successors(node))
    dist_to_outlet_m[node] = 0.0 if not succ else min(
        dist_to_outlet_m[s] + pixel_step_m(node, s) for s in succ
    )

# Number of upstream pixels
upstream_count = {}
for node in topo:
    upstream_count[node] = sum(upstream_count[p] + 1 for p in G_detail.predecessors(node))

# Which compressed edge each pixel belongs to
segments = []          # [u, v, pixel_edges, path_length_km]
segment_of = {}
for k, (u, v, data) in enumerate(G.edges(data=True)):
    path = data.get("original_path", [u, v])
    length_m = sum(pixel_step_m(path[i], path[i + 1]) for i in range(len(path) - 1))
    segments.append([int(u), int(v), len(path) - 1, round(length_m / 1000, 2)])
    for node in path[1:-1]:
        segment_of[node] = k

covered = len(segment_of) + G.number_of_nodes()
print(f"Pixels covered by compressed edges + kept nodes: {covered:,} / {G_detail.number_of_nodes():,}")
print(f"Total river length: {sum(s[3] for s in segments):,.0f} km")
print(f"Longest path to outlet: {max(dist_to_outlet_m.values()) / 1000:,.0f} km")

## 5. Build a searchable point table

In [ ]:
def coords(node, graph=None):
    data = (graph if graph is not None else G_detail).nodes[node]
    return float(data["latitude"]), float(data["longitude"])


def cluster_info(node, role):
    clusters = [c for c in G.nodes[node].get("usgs_clusters", []) if c.get("ml_role") == role]
    members = sorted({sid for c in clusters for sid in c.get("member_stations", [])})
    features = sorted({f for c in clusters for f in c.get("features", [])})
    return clusters, members, features


point_rows = []

def add_point(node, point_type, label, view, extra=None):
    lat, lon = coords(node)
    row = {"node_id": node, "type": point_type, "label": label, "view": view,
           "latitude": lat, "longitude": lon}
    row.update(extra or {})
    point_rows.append(row)

for node in sorted(source_nodes):
    add_point(node, "Network source", f"Source node {node}", "both")
for node in sorted(outlet_nodes):
    add_point(node, "Outlet / downstream leaf", f"Outlet node {node}", "both")
for node in sorted(junction_nodes):
    add_point(node, "Junction / meeting point", f"Junction node {node}", "both")

for role, nodes, label in [("train", train_nodes, "Training — 5/5"),
                           ("test_full", full_test_nodes, "Test — held-out 5/5"),
                           ("test_partial", partial_test_nodes, "Test — partial WQ")]:
    for node in sorted(nodes):
        _, members, _ = cluster_info(node, role)
        add_point(node, label, " / ".join(members) or f"node {node}", "both",
                  {"member_stations": ", ".join(members)})

for node in sorted(support_nodes):
    gauges = G.nodes[node].get("support_gauges", [])
    ids = sorted({str(g["station_id"]) for g in gauges if g.get("station_id") is not None})
    add_point(node, "Hydrology support", " / ".join(ids) or f"node {node}", "both",
              {"support_gauges": ", ".join(ids)})

for node in sorted(snapped_station_nodes):
    stations = G_detail.nodes[node]["stations"]
    label = " / ".join(f"{s['station_id']} {s.get('station_name', '')}".strip() for s in stations)
    add_point(node, "WQ station (snapped to pixel)", label, "uncompressed")

points_df = pd.DataFrame(point_rows)
points_df.to_csv(OUTPUT_POINTS_CSV, index=False)
print("Saved:", OUTPUT_POINTS_CSV)
print(points_df.groupby(["view", "type"]).size())

## 6. Create the interactive map with the compressed / uncompressed toggle

In [ ]:
# ==================================================
# CARTO BASEMAP API KEY
# (read from the CARTO_API_KEY environment variable, otherwise prompt)
# ==================================================

CARTO_API_KEY = os.environ.get("CARTO_API_KEY", "").strip()
if not CARTO_API_KEY:
    CARTO_API_KEY = getpass("Enter your CARTO Basemaps API key: ").strip()
if not CARTO_API_KEY:
    raise ValueError("A CARTO Basemaps API key is required.")


# ==================================================
# MAP
# ==================================================

all_lat = [float(d["latitude"]) for _, d in G_detail.nodes(data=True)]
all_lon = [float(d["longitude"]) for _, d in G_detail.nodes(data=True)]

m = folium.Map(
    location=[float(np.mean(all_lat)), float(np.mean(all_lon))],
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

OSM_ATTR = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
CARTO_ATTR = OSM_ATTR + ' &copy; <a href="https://carto.com/attribution/">CARTO</a>'

tile_osm = folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap", attr=OSM_ATTR, max_zoom=19, show=True,
).add_to(m)
tile_light = folium.TileLayer(
    tiles="https://basemaps.cartocdn.com/rastertiles/light_all/{z}/{x}/{y}{r}.png" f"?key={CARTO_API_KEY}",
    name="CARTO Positron", attr=CARTO_ATTR, max_zoom=20, show=False,
).add_to(m)
tile_dark = folium.TileLayer(
    tiles="https://basemaps.cartocdn.com/rastertiles/dark_all/{z}/{x}/{y}{r}.png" f"?key={CARTO_API_KEY}",
    name="CARTO Dark Matter", attr=CARTO_ATTR, max_zoom=20, show=False,
).add_to(m)


# ==================================================
# FEATURE GROUPS  (all start hidden; the toggle script shows the active view)
# ==================================================

def group(name):
    return folium.FeatureGroup(name=name, show=False).add_to(m)

# view -> list of (layer name, layer, visible by default)
VIEW_LAYERS = {"compressed": [], "uncompressed": []}

def register(views, name, layer, show=True):
    for v in views:
        VIEW_LAYERS[v].append((name, layer, show))

c_river = group("River network (compressed edges)")
c_nodes = group("Compressed nodes")
c_source = group("Network sources")
c_outlet = group("Outlets / downstream leaves")
c_junction = group("Junctions / meeting points")

u_river = group("River network (pixel chains)")
u_pixels = group("River-pixel nodes")
u_source = group("Network sources")
u_outlet = group("Outlets / downstream leaves")
u_junction = group("Junctions / meeting points")
u_snapped = group("WQ stations snapped to pixels")

train_layer = group("Training — 5/5")
full_test_layer = group("Test — held-out 5/5")
partial_test_layer = group("Test — partial WQ")
support_layer = group("Hydrology support")

register(["compressed"], "River network (compressed edges)", c_river)
register(["compressed"], "Compressed nodes (67)", c_nodes)
register(["compressed"], "Network sources", c_source)
register(["compressed"], "Outlets / downstream leaves", c_outlet)
register(["compressed"], "Junctions / meeting points", c_junction)

register(["uncompressed"], "River network (pixel chains)", u_river)
register(["uncompressed"], f"River-pixel nodes ({G_detail.number_of_nodes():,}, zoom ≥ {PIXEL_MIN_ZOOM})", u_pixels)
register(["uncompressed"], "Network sources", u_source)
register(["uncompressed"], "Outlets / downstream leaves", u_outlet)
register(["uncompressed"], "Junctions / meeting points", u_junction)
register(["uncompressed"], "WQ stations snapped to pixels", u_snapped)

for name, layer in [("Training — 5/5", train_layer), ("Test — held-out 5/5", full_test_layer),
                    ("Test — partial WQ", partial_test_layer), ("Hydrology support", support_layer)]:
    register(["compressed", "uncompressed"], name, layer)


# ==================================================
# POPUP HELPERS
# ==================================================

def google_maps_url(lat, lon):
    return f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"


def popup_html(node, title, details=None):
    lat, lon = coords(node)
    rows = [
        f"<b>{html.escape(str(title))}</b>",
        f"<b>Graph node:</b> {node}",
        f"<b>Latitude:</b> {lat:.6f}",
        f"<b>Longitude:</b> {lon:.6f}",
    ]
    for key, value in (details or {}).items():
        if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
            continue
        rows.append(f"<b>{html.escape(str(key))}:</b> {html.escape(str(value))}")
    rows.append(f'<a href="{google_maps_url(lat, lon)}" target="_blank">Open in Google Maps</a>')
    return folium.Popup("<br>".join(rows), max_width=420)


def pixel_details(node):
    d = G_detail.nodes[node]
    return {
        "Raster row / col": f"{d['row']} / {d['col']}",
        "Distance to outlet": f"{dist_to_outlet_m[node] / 1000:,.1f} km",
        "Upstream pixels": f"{upstream_count[node]:,}",
    }


def path_coords(path):
    # 5 decimals ≈ 1 m, plenty for ~90 m MERIT pixels and keeps the HTML small
    return [[round(c, 5) for c in coords(n)] for n in path if n in G_detail]


# ==================================================
# RIVER NETWORK — both views share the same geometry
# ==================================================

for k, (u, v, data) in enumerate(G.edges(data=True)):
    line = path_coords(data.get("original_path", [u, v]))
    if len(line) < 2:
        line = [list(coords(u)), list(coords(v))]
    _, _, pixel_edges, length_km = segments[k]

    folium.PolyLine(
        line, weight=3, opacity=0.7, color="#3388ff",
        tooltip=f"Compressed edge {u} → {v} | {data.get('length_m', 0) / 1000:.1f} km | "
                f"replaces {pixel_edges:,} pixel edges",
    ).add_to(c_river)

    folium.PolyLine(
        line, weight=2, opacity=0.85, color="#1e40af",
        tooltip=f"Pixel chain {u} … {v} | {pixel_edges + 1:,} pixel nodes | {length_km:.1f} km",
    ).add_to(u_river)


# ==================================================
# COMPRESSED NODES
# ==================================================

for node in G:
    roles = sorted({c.get("ml_role") for c in G.nodes[node].get("usgs_clusters", []) if c.get("ml_role")})
    folium.CircleMarker(
        list(coords(node)), radius=4, color="#1e3a8a", weight=1,
        fill=True, fill_color="#93c5fd", fill_opacity=0.95,
        tooltip=f"Compressed node {node}",
        popup=popup_html(node, "Compressed node", {
            "In-degree": G.in_degree(node),
            "Out-degree": G.out_degree(node),
            "ML roles": ", ".join(roles),
        }),
    ).add_to(c_nodes)


# ==================================================
# STRUCTURAL NODES (sources / outlets / junctions) — one set per view
# ==================================================

def coverage(node):
    distance = source_distance_lookup.get(node, np.nan)
    if pd.isna(distance):
        return "red", "No downstream measurement"
    color = "green" if distance <= 100 else "orange"
    return color, f"{distance:.1f} km to first downstream measurement"


for graph, sources, outlets, junctions, layers, extra in [
    (G, source_nodes, outlet_nodes, junction_nodes, (c_source, c_outlet, c_junction), lambda n: {}),
    (G_detail, d_source_nodes, d_outlet_nodes, d_junction_nodes, (u_source, u_outlet, u_junction), pixel_details),
]:
    s_layer, o_layer, j_layer = layers

    for node in sources:
        color, coverage_text = coverage(node)
        folium.Marker(
            list(coords(node)), tooltip=f"Source {node}",
            popup=popup_html(node, "Network source", {
                "Coverage": coverage_text,
                "In-degree": graph.in_degree(node),
                "Out-degree": graph.out_degree(node),
                **extra(node),
            }),
            icon=folium.Icon(color=color, icon="play", prefix="fa"),
        ).add_to(s_layer)

    for node in outlets:
        folium.Marker(
            list(coords(node)), tooltip=f"Outlet {node}",
            popup=popup_html(node, "Outlet / downstream leaf", {
                "In-degree": graph.in_degree(node),
                "Out-degree": graph.out_degree(node),
                **extra(node),
            }),
            icon=folium.Icon(color="darkgreen", icon="stop", prefix="fa"),
        ).add_to(o_layer)

    for node in junctions:
        folium.CircleMarker(
            list(coords(node)), radius=7, color="#f59e0b", weight=2,
            fill=True, fill_color="#f59e0b", fill_opacity=0.75,
            tooltip=f"Junction {node}",
            popup=popup_html(node, "Junction / meeting point", {
                "In-degree": graph.in_degree(node),
                "Out-degree": graph.out_degree(node),
                **extra(node),
            }),
        ).add_to(j_layer)


# ==================================================
# MONITORING ROLES (shared by both views)
# ==================================================

CORE_FEATURES = {"DO concentration", "Water temperature", "Specific conductance", "Turbidity", "pH"}

for role, nodes, layer, title, prefix, color, icon in [
    ("train", train_nodes, train_layer, "Training — 5/5", "TRAIN", "green", "circle"),
    ("test_full", full_test_nodes, full_test_layer, "Held-out test — 5/5", "FULL TEST", "red", "star"),
    ("test_partial", partial_test_nodes, partial_test_layer, "Partial WQ test", "PARTIAL TEST", "orange", "flask"),
]:
    for node in nodes:
        clusters, members, features = cluster_info(node, role)
        details = {"Stations": ", ".join(members), "Features": ", ".join(features)}
        if role == "test_partial":
            details = {
                "Stations": ", ".join(members),
                "Core features": max([c.get("feature_count", 0) for c in clusters], default=0),
                "Available": ", ".join(features),
                "Missing": ", ".join(sorted(CORE_FEATURES - set(features))),
            }
        details.update(pixel_details(node))
        folium.Marker(
            list(coords(node)),
            tooltip=f"{prefix}: " + (" / ".join(members) if members else str(node)),
            popup=popup_html(node, title, details),
            icon=folium.Icon(color=color, icon=icon, prefix="fa"),
        ).add_to(layer)

for node in support_nodes:
    gauges = G.nodes[node].get("support_gauges", [])
    gauge_ids = sorted({str(g["station_id"]) for g in gauges if g.get("station_id") is not None})
    used_by = sorted({s for g in gauges for s in g.get("used_by_wq_stations", [])})
    folium.Marker(
        list(coords(node)),
        tooltip="HYDROLOGY: " + (" / ".join(gauge_ids) if gauge_ids else str(node)),
        popup=popup_html(node, "Hydrology support", {
            "Gauge IDs": ", ".join(gauge_ids),
            "Used by WQ stations": ", ".join(used_by),
            **pixel_details(node),
        }),
        icon=folium.Icon(color="purple", icon="tint", prefix="fa"),
    ).add_to(support_layer)


# ==================================================
# WQ STATIONS SNAPPED TO THE PIXEL GRAPH (uncompressed only)
# ==================================================

for node in snapped_station_nodes:
    stations = G_detail.nodes[node]["stations"]
    for s in stations:
        details = {
            "Station": s.get("station_id"),
            "Name": s.get("station_name"),
            "Type": s.get("station_type"),
            "Snap distance": f"{s.get('distance_to_node_m', float('nan')):.0f} m",
            "Years": f"{s.get('first_year')}–{s.get('last_year')} ({s.get('years_with_data')} with data)",
            "Total results": s.get("total_results"),
            **pixel_details(node),
        }
        folium.Marker(
            [float(s.get("latitude", coords(node)[0])), float(s.get("longitude", coords(node)[1]))],
            tooltip=f"WQ station {s.get('station_id')}",
            popup=popup_html(node, "WQ station (snapped to pixel)", details),
            icon=folium.Icon(color="cadetblue", icon="vial", prefix="fa"),
        ).add_to(u_snapped)


# ==================================================
# SEARCH INDEX (points from both views)
# ==================================================

search_features = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(r.longitude), float(r.latitude)]},
        "properties": {"search_name": f"{r.type} | {r.label} | node {r.node_id}"},
    }
    for r in points_df.itertuples()
]

search_layer = folium.GeoJson(
    {"type": "FeatureCollection", "features": search_features},
    name="Search index",
    marker=folium.CircleMarker(radius=1, opacity=0, fill_opacity=0),
    show=False,
    control=False,
).add_to(m)

plugins.Search(
    layer=search_layer, search_label="search_name",
    placeholder="Search station, node, source, junction...",
    collapsed=False, position="topleft", zoom=12,
).add_to(m)


# ==================================================
# MAP CONTROLS
# ==================================================

plugins.Fullscreen(position="topleft", title="Fullscreen", title_cancel="Exit fullscreen").add_to(m)
plugins.MiniMap(toggle_display=True, minimized=False).add_to(m)
plugins.MousePosition(position="bottomright", separator=" | ", prefix="Lat/Lon:", num_digits=5).add_to(m)
plugins.MeasureControl(position="topleft", primary_length_unit="kilometers",
                       secondary_length_unit="miles").add_to(m)


# ==================================================
# LEGEND (rows tagged by view; the toggle shows/hides them)
# ==================================================

def legend_row(symbol, color, text, view="both"):
    return (f'<div class="rn-leg" data-view="{view}">'
            f'<span style="color:{color};">{symbol}</span> {text}</div>')

legend_html = (
    '<div id="rn-legend" style="position: fixed; bottom: 35px; left: 35px; z-index: 9999; '
    'background: rgba(255,255,255,0.94); border: 2px solid #777; border-radius: 8px; '
    'padding: 10px 12px; font-size: 13px; line-height: 1.45; '
    'box-shadow: 0 1px 6px rgba(0,0,0,0.25);">'
    '<b>River Network Legend</b> <span id="rn-legend-view" style="color:#555;"></span>'
    + legend_row("━", "#3388ff", "Compressed edge (follows river path)", "compressed")
    + legend_row("●", "#93c5fd", "Compressed node", "compressed")
    + legend_row("━", "#1e40af", "Pixel chain", "uncompressed")
    + legend_row("●", "#60a5fa", f"River-pixel node (zoom ≥ {PIXEL_MIN_ZOOM})", "uncompressed")
    + legend_row("■", "#5f9ea0", "WQ station snapped to pixel", "uncompressed")
    + legend_row("●", "#28a745", "Training — 5/5")
    + legend_row("★", "#d62728", "Test — held-out 5/5")
    + legend_row("◆", "#f59e0b", "Test — partial WQ")
    + legend_row("■", "#7e22ce", "Hydrology support")
    + legend_row("▶", "#16a34a", "Source: ≤100 km to measurement")
    + legend_row("▶", "#f59e0b", "Source: &gt;100 km to measurement")
    + legend_row("▶", "#dc2626", "Source: no downstream measurement")
    + legend_row("■", "#15803d", "Outlet / downstream leaf")
    + legend_row("●", "#f59e0b", "Junction / meeting point")
    + "</div>"
)
m.get_root().html.add_child(Element(legend_html))

toggle_css = """
<style>
.rn-view { background: rgba(255,255,255,0.96); padding: 8px 10px; border-radius: 8px;
           box-shadow: 0 1px 6px rgba(0,0,0,0.3); font: 13px/1.4 system-ui, sans-serif; min-width: 230px; }
.rn-view .rn-title { font-weight: 600; margin-bottom: 6px; }
.rn-seg { display: flex; border: 1px solid #1e40af; border-radius: 6px; overflow: hidden; }
.rn-seg button { flex: 1; border: 0; background: #fff; color: #1e40af; padding: 5px 8px;
                 font: 600 13px system-ui, sans-serif; cursor: pointer; }
.rn-seg button + button { border-left: 1px solid #1e40af; }
.rn-seg button.active { background: #1e40af; color: #fff; }
.rn-stats { margin-top: 6px; color: #333; }
.rn-note { margin-top: 4px; color: #6b7280; font-size: 12px; }
</style>
"""
m.get_root().header.add_child(Element(toggle_css))


# ==================================================
# UNCOMPRESSED PIXEL DATA + TOGGLE SCRIPT
# ==================================================

pixel_rows = []
for node, d in G_detail.nodes(data=True):
    pixel_rows.append([
        round(float(d["latitude"]), 5), round(float(d["longitude"]), 5), int(node),
        int(d["row"]), int(d["col"]), segment_of.get(node, -1),
        round(dist_to_outlet_m[node] / 1000, 1), upstream_count[node],
    ])


class JsBlock(MacroElement):
    """Raw JS rendered after every other map element (so all layer variables exist)."""
    _template = Template("{% macro script(this, kwargs) %}{{ this.code }}{% endmacro %}")

    def __init__(self, code):
        super().__init__()
        self._name = "JsBlock"
        self.code = code


def js_layers(view):
    return "[" + ",".join(
        f"[{json.dumps(name)}, {layer.get_name()}, {str(show).lower()}]"
        for name, layer, show in VIEW_LAYERS[view]
    ) + "]"


VIEW_STATS = {
    "compressed": f"{G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges",
    "uncompressed": f"{G_detail.number_of_nodes():,} river-pixel nodes · {G_detail.number_of_edges():,} edges",
}

toggle_js = r"""
(function () {
  var map = __MAP__;
  var P = __PIXELS__;   // [lat, lon, node, row, col, segment, dist_to_outlet_km, upstream_pixels]
  var S = __SEGMENTS__; // [u, v, pixel_edges, length_km]
  var MINZ = __MINZ__, MAXN = __MAXN__;
  var VIEWS = { compressed: __C_LAYERS__, uncompressed: __U_LAYERS__ };
  var STATS = __STATS__;
  var pixelLayer = __PIXEL_LAYER__;
  var current = null;

  // ---------- layer control (basemaps + active view's overlays) ----------
  var lc = L.control.layers(
    { "OpenStreetMap": __T_OSM__, "CARTO Positron": __T_LIGHT__, "CARTO Dark Matter": __T_DARK__ },
    {}, { collapsed: false, position: "topright" }
  );

  // ---------- view toggle control ----------
  var Toggle = L.Control.extend({
    options: { position: "topright" },
    onAdd: function () {
      var div = L.DomUtil.create("div", "rn-view");
      div.innerHTML =
        '<div class="rn-title">Network view</div>' +
        '<div class="rn-seg"><button data-v="compressed">Compressed</button>' +
        '<button data-v="uncompressed">Uncompressed</button></div>' +
        '<div class="rn-stats" id="rn-stats"></div><div class="rn-note" id="rn-note"></div>';
      L.DomEvent.disableClickPropagation(div);
      L.DomEvent.disableScrollPropagation(div);
      div.querySelectorAll("button").forEach(function (b) {
        b.addEventListener("click", function () { setView(b.getAttribute("data-v")); });
      });
      return div;
    }
  });
  new Toggle().addTo(map);
  lc.addTo(map);

  function setView(view) {
    if (view === current) return;
    Object.keys(VIEWS).forEach(function (v) {
      VIEWS[v].forEach(function (e) { lc.removeLayer(e[1]); if (map.hasLayer(e[1])) map.removeLayer(e[1]); });
    });
    VIEWS[view].forEach(function (e) {
      lc.addOverlay(e[1], e[0]);
      if (e[2]) map.addLayer(e[1]);
    });
    current = view;
    document.querySelectorAll(".rn-seg button").forEach(function (b) {
      b.classList.toggle("active", b.getAttribute("data-v") === view);
    });
    document.querySelectorAll(".rn-leg").forEach(function (r) {
      var v = r.getAttribute("data-view");
      r.style.display = (v === "both" || v === view) ? "" : "none";
    });
    document.getElementById("rn-stats").textContent = STATS[view];
    var lv = document.getElementById("rn-legend-view");
    if (lv) lv.textContent = "(" + view + ")";
    if (history.replaceState) history.replaceState(null, "", "#" + view);
    drawPixels();
  }

  // ---------- uncompressed pixel nodes: drawn only in view, when zoomed in ----------
  var renderer = L.canvas({ padding: 0.3 });

  function fmt(x) { return Number(x).toLocaleString(); }

  function pixelPopup(mk) {
    var p = P[mk._i], s = p[5] >= 0 ? S[p[5]] : null;
    var g = "https://www.google.com/maps/search/?api=1&query=" + p[0] + "," + p[1];
    return "<b>River-pixel node</b>" +
      "<br><b>Graph node:</b> " + p[2] +
      "<br><b>Latitude:</b> " + p[0].toFixed(5) +
      "<br><b>Longitude:</b> " + p[1].toFixed(5) +
      "<br><b>Raster row / col:</b> " + p[3] + " / " + p[4] +
      "<br><b>Distance to outlet:</b> " + fmt(p[6]) + " km" +
      "<br><b>Upstream pixels:</b> " + fmt(p[7]) +
      (s ? "<br><b>Folded into compressed edge:</b> " + s[0] + " → " + s[1] +
           " (" + fmt(s[2]) + " pixel edges, " + s[3] + " km)"
         : "<br><b>Kept as a node in the compressed graph</b>") +
      '<br><a href="' + g + '" target="_blank">Open in Google Maps</a>';
  }

  function drawPixels() {
    pixelLayer.clearLayers();
    var note = document.getElementById("rn-note");
    if (!note) return;
    if (current !== "uncompressed" || !map.hasLayer(pixelLayer)) { note.textContent = ""; return; }
    var z = map.getZoom();
    if (z < MINZ) {
      note.textContent = "Zoom in to level " + MINZ + "+ to see individual river-pixel nodes (now " + z + ").";
      return;
    }
    var b = map.getBounds().pad(0.15), n = 0, r = z >= 14 ? 4.5 : (z >= 13 ? 3.5 : 2.5);
    for (var i = 0; i < P.length; i++) {
      var p = P[i];
      if (!b.contains([p[0], p[1]])) continue;
      if (++n > MAXN) break;
      var mk = L.circleMarker([p[0], p[1]], {
        renderer: renderer, radius: r, color: "#1e3a8a", weight: 1,
        fillColor: p[5] < 0 ? "#facc15" : "#60a5fa", fillOpacity: 0.9
      });
      mk._i = i;
      mk.bindPopup(pixelPopup, { maxWidth: 420 });
      mk.bindTooltip("Pixel node " + p[2]);
      pixelLayer.addLayer(mk);
    }
    note.textContent = n > MAXN
      ? "Showing " + fmt(MAXN) + " river-pixel nodes in view — zoom in for all."
      : fmt(n) + " river-pixel nodes in view.";
  }

  map.on("moveend", drawPixels);
  map.on("overlayadd overlayremove", function (e) { if (e.layer === pixelLayer) drawPixels(); });

  var start = (location.hash || "").replace("#", "");
  setView(VIEWS[start] ? start : "compressed");
})();
"""

replacements = {
    "__MAP__": m.get_name(),
    "__PIXELS__": json.dumps(pixel_rows, separators=(",", ":")),
    "__SEGMENTS__": json.dumps(segments, separators=(",", ":")),
    "__MINZ__": str(PIXEL_MIN_ZOOM),
    "__MAXN__": str(PIXEL_MAX_IN_VIEW),
    "__C_LAYERS__": js_layers("compressed"),
    "__U_LAYERS__": js_layers("uncompressed"),
    "__STATS__": json.dumps(VIEW_STATS, ensure_ascii=False),
    "__PIXEL_LAYER__": u_pixels.get_name(),
    "__T_OSM__": tile_osm.get_name(),
    "__T_LIGHT__": tile_light.get_name(),
    "__T_DARK__": tile_dark.get_name(),
}
for key, value in replacements.items():
    toggle_js = toggle_js.replace(key, value)

m.add_child(JsBlock(toggle_js))

m.fit_bounds([[min(all_lat), min(all_lon)], [max(all_lat), max(all_lon)]])


# ==================================================
# SAVE
# ==================================================

m.save(OUTPUT_HTML)
print("Saved interactive map:", OUTPUT_HTML, f"({os.path.getsize(OUTPUT_HTML) / 1e6:.1f} MB)")

if SITE_HTML:
    m.save(SITE_HTML)
    print("Updated site:", SITE_HTML)

# The page is several MB, so it is not rendered inline here.
# Open the HTML file in a browser (or run the next cell to embed it).

In [ ]:
# Optional: preview inside the notebook (adds several MB to the .ipynb if you save it)
# from IPython.display import IFrame
# IFrame(OUTPUT_HTML, width="100%", height=700)

## 7. Optional quick summaries

In [ ]:
display(source_coverage.sort_values("distance_to_first_measurement_km", ascending=False, na_position="first"))

In [ ]:
display(points_df.sort_values(["view", "type", "label"]))